# Needs to be appended at the end of the second pipe. Worked as of april 1st 2026.

Used to assign predictions and sentiment probabiltiy with MPL model to all sentences from all immigration books, then attach the labels previous given to them. 

In [ ]:
"""
process_sentiments.py

1. Loads all sentence CSVs from dataframes/untagged/sentence_dfs/
2. Runs the MLP sentiment model on every sentence
3. Exports full scored CSV
4. Merges labeled data from dataframes/tagged/scheme_two/t_extreme_sentences_partial/
   onto the scored dataframe (left join → labels are NaN where not tagged)
5. Exports merged CSV
"""

import pandas as pd
import numpy as np
import torch
from transformers import DistilBertModel, DistilBertTokenizer
import os
import glob

# ── Load DistilBERT backbone ───────────────────────────────────────────────────
print("Loading DistilBERT tokenizer and backbone …")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
distilbert.eval()

# Use the MLP model defined earlier in the notebook
mlp_model.eval()

LABELS = ["Positive", "Negative", "Neutral"]

# ── Encoder (mirrors your existing function, adds batching) ───────────────────
def encode_sentences(sentences, max_length=128):
    """Return token embeddings (n, seq_len, 768) as a numpy array."""
    inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    with torch.no_grad():
        token_embeddings = distilbert(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        ).last_hidden_state
    return token_embeddings.numpy()


# ── Predict (no manual labels required) ───────────────────────────────────────
def predict_sentences(sentences, model, batch_size=32, max_length=128):
    """
    Run the sentiment head over `sentences` in batches.

    Returns a DataFrame with columns:
        sentence, Positive_score, Positive_pred,
                  Negative_score, Negative_pred,
                  Neutral_score,  Neutral_pred
    """
    all_probs = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start : start + batch_size]
        embeddings = torch.tensor(
            encode_sentences(batch, max_length=max_length),
            dtype=torch.float32,
        )
        with torch.no_grad():
            logits = model(embeddings)
            probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        print(f"  scored {min(start + batch_size, len(sentences))}/{len(sentences)}", end="\r")

    print()
    all_probs = np.vstack(all_probs)  # (n_sentences, 3)
    preds = (all_probs >= 0.5).astype(int)

    result = pd.DataFrame({"sentence": sentences})
    for i, label in enumerate(LABELS):
        result[f"{label}_score"] = all_probs[:, i]
        result[f"{label}_pred"] = preds[:, i]

    return result


# ── 1. Load all untagged sentence CSVs ────────────────────────────────────────
sentence_dir = "dataframes/untagged/sentence_dfs"
print(f"\nLoading sentence CSVs from {sentence_dir} …")
sentence_files = glob.glob(os.path.join(sentence_dir, "*.csv"))
if not sentence_files:
    raise FileNotFoundError(f"No CSVs found in {sentence_dir}")

dfs = []
for f in sorted(sentence_files):
    df = pd.read_csv(f)
    print(f"  {os.path.basename(f)}: {len(df)} rows")
    dfs.append(df)

sentences_df = pd.concat(dfs, ignore_index=True)
print(f"Total sentences: {len(sentences_df)}")

# ── 2. Run sentiment model ─────────────────────────────────────────────────────
print("\nRunning sentiment model …")
scores_df = predict_sentences(
    sentences_df["sentence"].tolist(),
    mlp_model,
    batch_size=32,
    max_length=128,
)

# Attach metadata columns (book_name, chapter_id, entry_id, date) to scores
meta_cols = [c for c in ["book_name", "chapter_id", "entry_id", "date"] if c in sentences_df.columns]
full_scored = pd.concat(
    [sentences_df[meta_cols].reset_index(drop=True), scores_df],
    axis=1,
)

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
scored_path = os.path.join(output_dir, "all_sentences_scored.csv")
full_scored.to_csv(scored_path, index=False)
print(f"Saved scored sentences → {scored_path}")


# ── 3. Load all labeled CSVs ───────────────────────────────────────────────────
labeled_dir = "dataframes/tagged/scheme_two/t_extreme_sentences_partial"
print(f"\nLoading labeled CSVs from {labeled_dir} …")
labeled_files = glob.glob(os.path.join(labeled_dir, "*.csv"))
if not labeled_files:
    raise FileNotFoundError(f"No labeled CSVs found in {labeled_dir}")

label_dfs = []
for f in sorted(labeled_files):
    df = pd.read_csv(f)
    print(f"  {os.path.basename(f)}: {len(df)} rows")
    label_dfs.append(df)

labeled_df = pd.concat(label_dfs, ignore_index=True)
print(f"Total labeled rows: {len(labeled_df)}")

# Keep only the label columns + join keys from the labeled data
LABEL_COLS = ["positive", "negative", "neutral", "immigration"]
JOIN_KEYS = ["book_name", "chapter_id", "entry_id", "sentence"]

# Use whichever join keys exist in both dataframes
available_keys = [k for k in JOIN_KEYS if k in full_scored.columns and k in labeled_df.columns]
print(f"Joining on: {available_keys}")

labeled_slim = labeled_df[available_keys + [c for c in LABEL_COLS if c in labeled_df.columns]].copy()

# ── 4. Left-join labels onto scored dataframe ─────────────────────────────────
merged_df = full_scored.merge(labeled_slim, on=available_keys, how="left")

labeled_count = merged_df["positive"].notna().sum() if "positive" in merged_df.columns else "N/A"
print(f"Matched {labeled_count} labeled sentences out of {len(merged_df)}")

merged_path = os.path.join(output_dir, "all_sentences_scored_with_labels.csv")
merged_df.to_csv(merged_path, index=False)
print(f"Saved merged output → {merged_path}")

print("\nDone! Output files:")
print(f"  {scored_path}          ← all sentences + model scores")
print(f"  {merged_path}  ← same + ground-truth labels where available")

In [ ]:
import pandas as pd
import os

# Load the CSV file into a DataFrame
input_file = "output/all_sentences_scored_with_labels.csv"  # Update the path if needed
output_folder = "output/books_with_labels"  # Folder to store the separated files

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Read the CSV file
df = pd.read_csv(input_file)

# Group by 'book_name' and save each group as a separate CSV
for book_name, group in df.groupby("book_name"):
    # Create a filename for each book
    output_file = os.path.join(output_folder, f"{book_name}_sentences.csv")
    
    # Save the group to a CSV file
    group.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

print("All books have been separated and saved.")